In [3]:
!pip install transformers torch -q

from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer
import torch
from threading import Thread

model_name = "mzoelfakar/Al-Khwarizmi-3B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.bfloat16, device_map="auto")

SYSTEM_PROMPT = (
    "You are Al-Khwarizmi, a math tutor named after Muhammad ibn Musa al-Khwarizmi, the mathematician who lived in the 9th century and whose name is the direct origin of the word 'algorithm'. "
    "You're a fine-tuned version of HuggingFaceTB/SmolLM3-3B-Base. "
    "Solve problems step by step."
)

history = []

print("Chat with Al-Khwarizmi-3B (type 'exit' to quit)\n")

while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        break

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for u, a in history:
        messages.append({"role": "user", "content": u})
        messages.append({"role": "assistant", "content": a})
    messages.append({"role": "user", "content": user_input})

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    generation_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=300,
        temperature=0.7,
        do_sample=True,
    )

    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    print("Al-Khwarizmi: ", end="", flush=True)
    response = ""
    for new_text in streamer:
        print(new_text, end="", flush=True)
        response += new_text
    print("\n")

    history.append((user_input, response))

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

Chat with Al-Khwarizmi-3B (type 'exit' to quit)

You: Hello
Al-Khwarizmi: Hello! How can I help you today?

You: 2+2=?
Al-Khwarizmi: 2+2=<<4=4>>4



KeyboardInterrupt: Interrupted by user